<a href="https://colab.research.google.com/github/ssk-algoverse/sae-binding/blob/main/sae/dict_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformer-lens dictionary-learning

### Load the model

In [3]:
import numpy as np
import pandas as pd

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
import torch
from huggingface_hub import hf_hub_download
from transformer_lens import HookedTransformer, HookedTransformerConfig
import numpy as np
import pandas as pd
import ast
from torch.utils.data import Dataset, DataLoader

In [ ]:
from huggingface_hub import hf_hub_download
from IPython.display import clear_output


REPO_ID = "sojup/entity_binding_test"
FILENAME = "D256_L3_H2_attnOnly1_lr5.0e-04_wd0.01.pt"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
clear_output()

In [ ]:
# import numpy as np
# import pandas as pd
# from tqdm import tqdm

E = 100  # num entities
T = 10   # num types/relations

SEP = E + T
Q = E + T + 1
PAD = E + T + 2
D_VOCAB = E + T + 3

IGNORE_INDEX = -100
# ENTITIES = np.arange(0, E)
# TYPES    = np.arange(E, E + T)

# N_WORLDS = 80_000
# MIN_FACTS, MAX_FACTS = 4, 8
# SEED = 0

# rng = np.random.default_rng(SEED)

# def produce_example(num_relations: int, *, allow_self_loops: bool = False):
#     facts = []
#     seen_head_rel = set()
#     seen_e = set()
#     seen_t = set()

#     while len(facts) < num_relations:
#         e = int(rng.integers(0, E))
#         t = int(TYPES[rng.integers(0, T)])

#         # enforce uniqueness of e and t (not just the tuple)
#         if e in seen_e or t in seen_t:
#             continue
#         if (e, t) in seen_head_rel:
#             continue

#         # forbid self-loop (optional)
#         e2 = int(rng.integers(0, E))
#         while not allow_self_loops and e2 == e:
#             e2 = int(rng.integers(0, E))

#         seen_head_rel.add((e, t))
#         seen_e.add(e)
#         seen_t.add(t)
#         facts.append((e, t, e2))

#     q_idx = int(rng.integers(0, num_relations))
#     Eq, Tq, E2q = facts[q_idx]

#     seq = []
#     for (e, t, e2) in facts:
#         seq.extend([e, t, e2, SEP])
#     seq.extend([Tq, Eq, Q])

#     return seq, E2q, facts



# rows = []
# for _ in tqdm(range(N_WORLDS)):
#     k = int(rng.integers(MIN_FACTS, MAX_FACTS + 1))
#     seq, label, _ = produce_example(k, allow_self_loops=False)
#     rows.append({"tokens": seq, "label": label})

# df = pd.DataFrame(rows)

In [ ]:
N_LAYERS = 3
HEADS = 2

d_model = 256
d_mlp   = 1024
n_ctx   = 64
lr = 3e-4
betas = (0.9, 0.98)
weight_decay = 0.01
num_epochs = 30

def build_model(n_layers: int, n_heads: int) -> HookedTransformer:
    if d_model % n_heads != 0:
        return None
    d_head = d_model // n_heads

    cfg = HookedTransformerConfig(
        n_layers=n_layers,
        n_heads=n_heads,
        d_model=d_model,
        d_head=d_head,
        d_mlp=d_mlp,
        n_ctx=n_ctx,
        d_vocab=D_VOCAB,
        d_vocab_out=E,
        act_fn="gelu",
        attn_only=True,
        normalization_type="LN",
    )
    return HookedTransformer(cfg)

model = build_model(N_LAYERS, HEADS)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Load the model
# Create a new model instance with the same configuration
pretrained_weights = torch.load(weights_path, map_location=device, weights_only=True)
state_dict = pretrained_weights["model"]
model.load_state_dict(state_dict)

print("Model loaded successfully.")

In [ ]:
# from sklearn.model_selection import train_test_split

# train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
# val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
# from sklearn.model_selection import train_test_split
# Removed redundant import: from datasets.arrow_dataset import Dataset # Moved import here to avoid circular dependency
from torch.utils.data import Dataset # Ensure torch.utils.data.Dataset is used
import torch
import ast

class EntityBindingDataset(Dataset):
    def __init__(self, dataframe, parse_tokens_if_str=True):
        self.df = dataframe.reset_index(drop=True)
        self.parse_tokens_if_str = parse_tokens_if_str

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row["tokens"]
        # Ensure seq is a list if it was read as a string (e.g., from CSV without converters)
        if isinstance(seq, str) and self.parse_tokens_if_str:
            seq = ast.literal_eval(seq)
        # Convert the list of integers to a torch tensor
        tokens = torch.tensor(seq, dtype=torch.long)
        label  = torch.tensor(int(row["label"]), dtype=torch.long)
        return tokens, label

# test_df = pd.read_csv('test_df.csv', converters={"tokens": ast.literal_eval})
# test_dataset = EntityBindingDataset(test_df)
# train_dataset = EntityBindingDataset(train_df)
# val_dataset = EntityBindingDataset(val_df)

In [ ]:

from datasets import load_dataset

train_dataset = load_dataset("sojup/entity_binding", split="train")
train_df = train_dataset.to_pandas()
train_dataset = EntityBindingDataset(train_df)


val_dataset = load_dataset("sojup/entity_binding", split="validation")
val_df = val_dataset.to_pandas()
val_dataset = EntityBindingDataset(val_df)


test_dataset = load_dataset("sojup/entity_binding", split="test")
test_df = test_dataset.to_pandas()
test_dataset = EntityBindingDataset(test_df)

In [ ]:
def collate_fn(batch):
    max_len = max(len(seq) for seq,_ in batch)
    B = len(batch)
    toks   = torch.full((B, max_len), PAD, dtype=torch.long)
    target = torch.full((B, max_len), IGNORE_INDEX, dtype=torch.long)

    for i, (seq, label) in enumerate(batch):
        x = torch.tensor(seq if isinstance(seq, list) else seq.tolist(), dtype=torch.long)
        L = len(x)
        toks[i, :L] = x
        q_pos = (x == Q).nonzero(as_tuple=False).squeeze()
        assert q_pos.numel() == 1, "Each example must have exactly one Q"
        target[i, q_pos.item()] = int(label)
    return toks, target

### Train a SAE

In [ ]:
import torch as t
from dictionary_learning.trainers.standard import StandardTrainer

In [ ]:
max(len(seq) for seq,_ in train_dataset), max(len(seq) for seq,_ in val_dataset)

In [22]:
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import trange

train_loader = DataLoader(
    train_dataset, # Use the train_dataset created in the previous cell
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset, # Use the val_dataset created in the previous cell
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
)

# ---- Choose which activations to train the SAE on ----
# Common choices: "blocks.{L}.hook_resid_pre", "blocks.{L}.hook_resid_post", "blocks.{L}.attn.hook_z", "blocks.{L}.hook_mlp_out" (if MLP exists)
HOOK_POINT = "blocks.1.hook_resid_pre"   # <- change layer if you like

# ---- Minimal SAE (ReLU + tied decoder option) ----
class SAE(nn.Module):
    def __init__(self, d_in, d_sae, tied_decoder=True):
        super().__init__()
        self.d_in = d_in
        self.d_sae = d_sae
        self.enc = nn.Linear(d_in, d_sae, bias=True)
        self.act = nn.ReLU()
        self.dec = nn.Linear(d_sae, d_in, bias=True)
        self.tied_decoder = tied_decoder

        # Init: small weights help sparsity
        nn.init.kaiming_uniform_(self.enc.weight, a=math.sqrt(5))
        nn.init.zeros_(self.enc.bias)
        nn.init.xavier_uniform_(self.dec.weight)
        nn.init.zeros_(self.dec.bias)

    def forward(self, x):
        h = self.act(self.enc(x))
        if self.tied_decoder:
            # Use encoder weights transposed (classic tied weights); keep decoder bias
            recon = torch.matmul(h, self.enc.weight) + self.dec.bias
        else:
            recon = self.dec(h)
        return recon, h

# ---- Probe one batch to infer d_in from the hook ----
model.eval()
with torch.no_grad():
    test_tokens, _ = next(iter(train_loader))
    _, cache = model.run_with_cache(test_tokens)
    sample_act = cache[HOOK_POINT]  # [B, pos, d_model]
    d_in = sample_act.shape[-1]
    del cache

print(f"SAE input dim (d_in) = {d_in}")

# ---- Hyperparams for SAE ----
d_sae         = 4096          # dictionary size (tweak to taste)
l1_coeff      = 3e-3          # sparsity penalty (try 1e-3 to 1e-2)
sae_lr        = 1e-3
sae_wd        = 1e-4
sae_epochs    = 3             # each epoch = full pass over token batches (activations are sampled on the fly)
max_positions = None          # optionally limit positions per batch (e.g., 256) to reduce memory

sae = SAE(d_in=d_in, d_sae=d_sae, tied_decoder=True).to(device)
opt = torch.optim.AdamW(sae.parameters(), lr=sae_lr, weight_decay=sae_wd)

# ---- Training loop (stream activations on-the-fly) ----
def batch_acts_from_hook(tokens):
    """
    Returns a 2D tensor of activations [N_nonpad, d_in] from the chosen hook,
    only at non-PAD positions.
    """
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens)
        acts = cache[HOOK_POINT]                      # [B, pos, d_in]
        mask = (tokens != PAD).unsqueeze(-1)          # [B, pos, 1]
        acts = acts[mask.expand_as(acts)]             # [N_nonpad * d_in] flattened
        acts = acts.view(-1, d_in)                    # [N_nonpad, d_in]
        if max_positions is not None and acts.shape[0] > max_positions:
            # random subset to cap compute
            idx = torch.randperm(acts.shape[0], device=acts.device)[:max_positions]
            acts = acts[idx]
        return acts

def eval_epoch(loader):
    sae.eval()
    total_mse, total_l1, total_n = 0.0, 0.0, 0
    with torch.no_grad():
        for tokens, _ in loader:
            X = batch_acts_from_hook(tokens)
            recon, h = sae(X)
            mse = torch.mean((recon - X)**2)
            l1  = torch.mean(torch.abs(h))
            n   = X.shape[0]
            total_mse += mse.item() * n
            total_l1  += l1.item()  * n
            total_n   += n
    return (total_mse/total_n), (total_l1/total_n)

print("Starting SAE training…")
for epoch in range(1, sae_epochs+1):
    sae.train()
    pbar = trange(len(train_loader), leave=False)
    running_mse = running_l1 = 0.0
    count = 0

    for tokens, _ in train_loader:
        X = batch_acts_from_hook(tokens)    # [N, d_in]
        recon, h = sae(X)

        mse = torch.mean((recon - X)**2)
        l1  = torch.mean(torch.abs(h))
        loss = mse + l1_coeff * l1

        opt.zero_grad()
        loss.backward()
        opt.step()

        running_mse += mse.item() * X.shape[0]
        running_l1  += l1.item()  * X.shape[0]
        count       += X.shape[0]

        pbar.set_description(
            f"epoch {epoch} | mse {running_mse/max(1,count):.5f} | l1 {running_l1/max(1,count):.5f}"
        )
        pbar.update(1)
    pbar.close()

    val_mse, val_l1 = eval_epoch(val_loader)
    print(f"[epoch {epoch}] val_mse={val_mse:.6f}  val_l1={val_l1:.6f}")

# ---- Save the trained SAE ----
sae_ckpt = {
    "config": {
        "d_in": d_in,
        "d_sae": d_sae,
        "tied_decoder": True,
        "hook_point": HOOK_POINT,
        "l1_coeff": l1_coeff,
        "sae_lr": sae_lr,
        "sae_wd": sae_wd,
        "epochs": sae_epochs,
    },
    "state_dict": sae.state_dict(),
}
save_path = "sae_resid_pre_L1.pt"
torch.save(sae_ckpt, save_path)
print(f"Saved SAE to {save_path}")

SAE input dim (d_in) = 256
Starting SAE training…


epoch 1 | mse 1.03297 | l1 0.47773:  16%|█▌        | 316/2000 [01:49<08:41,  3.23it/s]

KeyboardInterrupt: 